<a href="https://colab.research.google.com/github/diazcas2/mIArmario/blob/main/MODULO1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install torch transformers google-generativeai langdetect --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 35.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [7]:
import json
import re
import time
import torch
import google.generativeai as genai
from transformers import pipeline
from google.colab import userdata

In [8]:
# Pon tu API key de Gemini aquí
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
# Whisper: float16 si hay GPU, float32 si no
_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
whisper = pipeline(
    "automatic-speech-recognition",
    "openai/whisper-large-v3",
    dtype=_dtype
)

In [5]:
def transcribir_voz(ruta_audio):

    resultado = whisper(ruta_audio)
    return resultado["text"].strip()


def llamar_gemini(prompt, reintentos=3):

    for intento in range(reintentos):
        try:
            respuesta = model.generate_content(prompt)
            return respuesta.text
        except Exception as e:
            if "429" in str(e):
                espera = 35 * (intento + 1)
                print(f"Límite alcanzado, esperando {espera}s...")
                time.sleep(espera)
            else:
                raise e
    raise Exception("Se agotaron los reintentos de Gemini")


def normalizar_texto(texto_crudo):

    prompt = f"""
      Eres el preprocesador de un asistente de moda personal.
      Recibirás una frase del usuario describiendo qué outfit quiere.
      Tu tarea es:
      1. Corregir errores ortográficos o de transcripción de voz.
      2. Eliminar muletillas, ruidos o palabras sin sentido.
      3. Reformular la frase de forma clara y concisa, manteniendo la intención original.
      4. NO añadas información que el usuario no haya dado.

      Responde ÚNICAMENTE con la frase normalizada, sin explicaciones ni texto extra.

      Frase original: {texto_crudo}
      """
    return llamar_gemini(prompt).strip()


def detectar_idioma(texto):

    try:
        from langdetect import detect
        return detect(texto)
    except Exception:
        return "es"


In [ ]:
def modulo1_entrada(fuente, texto=None, ruta_audio=None):
    """
    Punto de entrada del Módulo 1.

    Acepta texto directo o audio, normaliza el input con Gemini T2T
    y devuelve el JSON estandarizado para el Módulo 3.

    Args:
        fuente (str)     : 'texto' | 'voz'
        texto (str)      : Texto escrito por el usuario  (si fuente='texto')
        ruta_audio (str) : Ruta al archivo de audio      (si fuente='voz')

    Returns:
        dict: JSON contrato Módulo 1 → Módulo 3:
              {
                  "texto"  : str,
                  "fuente" : str,
                  "idioma" : str
              }
    """

    # Paso 1: Obtener texto crudo según la fuente
    if fuente == "voz":
        texto_crudo = transcribir_voz(ruta_audio)

    elif fuente == "texto":
        texto_crudo = texto.strip()

    else:
        raise ValueError(f"'fuente' debe ser 'texto' o 'voz', recibido: '{fuente}'")

    # Paso 2: Normalizar con Gemini T2T
    texto_normalizado = normalizar_texto(texto_crudo)

    # Paso 3: Detectar idioma
    idioma = detectar_idioma(texto_normalizado)

    # Paso 4: Construir JSON de salida
    salida = {
        "texto": texto_normalizado,
        "fuente": fuente,
        "idioma": idioma
    }

    return salida


In [ ]:
# Ejemplo 1 — Entrada por texto
resultado = modulo1_entrada(
    fuente="texto",
    texto="quiero algo elegante pa una cena esta noche porfa"
)

print(json.dumps(resultado, ensure_ascii=False, indent=2))

In [ ]:
# Ejemplo 2 — Entrada por voz
resultado = modulo1_entrada(
    fuente="voz",
    ruta_audio="mi_audio.mp3"   # <-- cambia por tu archivo
)

print(json.dumps(resultado, ensure_ascii=False, indent=2))